# Make a split

A split decides **which charts to keep** and **which part (train/validation/test) each one goes to**. It scans every `.tja` under `raw/`, drops charts the parser can't handle or the filters exclude, and splits the rest by audio file, so Oni and Ura of one song always land in the same part.

Output goes to `splits/<name>/`: `config.json`, `manifest.csv` and `report.json`. Splits are never edited in place. If you change a filter, give the split a new name.

In [ ]:
from donderformer.paths import DataPaths
from donderformer.data.data_splitter import DataSplitter

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for the toy dataset.
paths = DataPaths()

splitter = DataSplitter(
    name="oni_v1",
    courses=("oni",),          # None = all; options: easy, normal, hard, oni, ura
    genres=None,               # None = all; top-level ESE folders, e.g. ("01 Pop", "02 Anime")
    bpm_range=None,            # e.g. (100, 220)
    level_range=None,          # e.g. (7, 10)
    ratios=(0.8, 0.1, 0.1),
    seed=42,
)

In [ ]:
report = splitter.run(paths)

## Report

Re-run from here to plot any existing split's `report.json`.

In [ ]:
import json
import matplotlib.pyplot as plt

report = json.loads((paths.split_dir(splitter.name) / "report.json").read_text())

panels = [("Kept per split", report["kept_by"]["split"]),
          ("Kept per course", report["kept_by"]["course"]),
          ("Kept per genre", report["kept_by"]["genre"])]
if "lost_reasons" in report:
    panels.append(("Lost reasons (top 10)", dict(list(report["lost_reasons"].items())[:10])))

fig, axes = plt.subplots(1, len(panels) + 1, figsize=(5 * (len(panels) + 1), 4))
for ax, (title, counts) in zip(axes, panels):
    ax.bar(list(counts), list(counts.values()))
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=60)
axes[-1].hist(report["kept_bpm"], bins=30)
axes[-1].set_title("BPM of kept charts")
fig.tight_layout()
plt.show()